# F1-scientific-python — Practice p21 — Solution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 20260804
rng = np.random.default_rng(SEED)

**Task A.** Implement exactly `def hist_counts(values, edges):`.

- `values`: 1-D float array; `edges`: 1-D array of `k + 1` increasing bin
  boundaries.
- Return an integer array of shape `(k,)` where entry `j` counts the values
  `v` with `edges[j] <= v < edges[j+1]` — except the **last** bin, which also
  includes `v == edges[-1]` (this matches `np.histogram`).

Hint: give `edges[:-1]` and `edges[1:]` a size-1 **column** axis so each
compares against all of `values` at once — two `(k, n)` masks, combined, then
collapsed along the right axis. Handle the last-bin edge case with one extra
mask count added to the final entry.

In [ ]:
def hist_counts(values, edges):
    lo = edges[:-1][:, None]                  # (k, 1)
    hi = edges[1:][:, None]                   # (k, 1)
    in_bin = (values >= lo) & (values < hi)   # (k, n): value j in bin row?
    counts = in_bin.sum(axis=1)               # collapse the values axis
    counts[-1] += (values == edges[-1]).sum() # right edge belongs to last bin
    return counts

Broadcasting `(k, 1)` boundaries against `(n,)` values builds a `(k, n)` grid answering "is value `v` inside bin `j`?" for every pair at once; collapsing `axis=1` counts per bin. The double loop this replaces — over bins and values — never gets written.

**Task B.** Check your function against NumPy's own counter. With a seeded
generator (`SEED = 20260804`), draw **`draws`** — 1000 values from
`rng.random` — and set `edges = np.linspace(0, 1, 11)`. Compute
**`mine`** = `hist_counts(draws, edges)` and **`ref`** =
`np.histogram(draws, bins=edges)[0]`, and show they are identical. Also test
the right-edge rule on the handmade array
`np.array([0.0, 0.5, 1.0])` with the same edges.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
draws = rng.random(1000)
edges = np.linspace(0, 1, 11)

mine = hist_counts(draws, edges)
ref = np.histogram(draws, bins=edges)[0]
print(mine)
print("matches np.histogram:", np.array_equal(mine, ref))

tricky = np.array([0.0, 0.5, 1.0])
print(hist_counts(tricky, edges))     # the 1.0 must land in the LAST bin

Seeding makes the comparison meaningful: every run tests the same 1000 values. The `tricky` case pins down the boundary rule — `1.0` equals the final edge and must be counted in bin 9, not dropped.

**Task C.** Draw the picture twice and confirm they agree: a labeled
`plt.hist` of `draws` with `bins=edges`, and (second figure) a labeled plot
of your own `mine` counts drawn with `plt.plot(centers, mine)` where
**`centers`** is the midpoint of each bin (compute it from `edges` with one
shifted-slice expression — no loops).

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(draws, bins=edges)
plt.title("1000 seeded draws (plt.hist)")
plt.xlabel("value")
plt.ylabel("how many draws")
plt.show()

centers = (edges[:-1] + edges[1:]) / 2
plt.figure(figsize=(7, 4))
plt.plot(centers, mine, label="my hist_counts")
plt.title("Same counts, computed by hand")
plt.xlabel("bin center")
plt.ylabel("how many draws")
plt.legend()
plt.show()

The shifted-slice midpoint trick pairs each edge with its successor — the same idiom as p15's consecutive differences. The line plot of your counts traces exactly the tops of `plt.hist`'s bars, which is the visual proof the two computations agree.

### Answer check

In [ ]:
SEED = 20260804
draws_ref = np.random.default_rng(SEED).random(1000)
assert np.array_equal(draws, draws_ref)
edges = np.linspace(0, 1, 11)
assert np.array_equal(mine, np.histogram(draws, bins=edges)[0])
assert mine.sum() == 1000
assert np.array_equal(hist_counts(np.array([0.0, 0.5, 1.0]), edges),
                      np.histogram(np.array([0.0, 0.5, 1.0]), bins=edges)[0])
assert np.array_equal(centers, (edges[:-1] + edges[1:]) / 2)
rng2 = np.random.default_rng(7)
v2 = rng2.random(313) * 4 - 2
e2 = np.linspace(-2, 2, 9)
assert np.array_equal(hist_counts(v2, e2), np.histogram(v2, bins=e2)[0])
print("all checks passed")